In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_json('data/script-bag-of-words.json')

In [3]:
df.tail()

,episodeAlt,seasonNum,episodeNum,episodeTitle,text
68,S8E2,8,2,A Knight of the Seven Kingdoms,"[{'name': 'Daenerys Targaryen', 'text': 'About..."
69,S8E3,8,3,The Long Night,"[{'name': 'Northman #1', 'text': 'Oi!'}, {'nam..."
70,S8E4,8,4,The Last of the Starks,"[{'name': 'Jon Snow', 'text': 'And Everyone It..."
71,S8E5,8,5,The Bells,"[{'name': 'Lord Varys', 'text': 'And? Come Not..."
72,S8E6,8,6,The Iron Throne,"[{'name': 'Tyrion Lannister', 'text': 'I'll fi..."


In [4]:
df.iloc[-1]['text']

[{'name': 'Tyrion Lannister', 'text': "I'll find later. you"},
 {'name': 'Jon Snow', 'text': "It's Let me men not safe. send some with you."},
 {'name': 'Tyrion Lannister', 'text': "I'm alone. going"},
 {'name': 'Grey Worm',
  'text': 'Daenerys I In Queen, Targaryen, die. name of one sentence the the to true you'},
 {'name': 'Jon Snow',
  'text': "Grey It's These Worm! are men over. prisoners."},
 {'name': 'Grey Worm',
  'text': "It Queen's are defeated. enemies is not over the until"},
 {'name': 'Davos Seaworth',
  'text': "How They're be? defeated do knees. more much on their them to want you"},
 {'name': 'Grey Worm', 'text': 'They are breathing.'},
 {'name': 'Davos Seaworth', 'text': 'Look We around friend. won. you,'},
 {'name': 'Grey Worm', 'text': "I commands, my not obey queen's yours."},
 {'name': 'Jon Snow', 'text': "And Queen's are commands? the what"},
 {'name': 'Grey Worm',
  'text': '"Kill Cersei Lannister." These They all are chose fight follow for free her. men. to who'}

In [5]:
dialouge = {}
for index, row in df.iterrows():
    for item in row['text']:
        if item['name'] in dialouge:
            # append
            dialouge[item['name']] = dialouge[item['name']] + item['text']
        else:
            # create character
            dialouge[item['name']] = item['text'] + " "

In [6]:
len(dialouge)

817

In [7]:
new_df = pd.DataFrame()
new_df['character'] = dialouge.keys()
new_df['words'] = dialouge.values()

In [8]:
new_df.iloc[:,0:3].head()

,character,words
0,Will,"Easy, boy. I've I've Wildlings a a do ever in ..."
1,Waymar Royce,One They're What a and another before d'you ea...
2,Gared,Wall. We back head should the to Our They We W...
3,Jon Snow,Father's Go on. watching. And mother. yourBran...
4,Septa Mordane,"Fine Well always. as done. work, I Quite beaut..."


In [9]:
new_df['num_words'] = new_df['words'].apply(lambda x:len(x.split()))

In [10]:
new_df = new_df.sort_values('num_words',ascending=False)

In [11]:
new_df = new_df.head(150)

In [12]:
new_df.shape

(150, 3)

In [13]:
new_df.head()

,character,words,num_words
17,Tyrion Lannister,It Mmh. Northern about girls. is say the they ...,25924
13,Cersei Lannister,And And Casterly One Rock. When about afraid. ...,14294
3,Jon Snow,Father's Go on. watching. And mother. yourBran...,11488
20,Daenerys Targaryen,We've a and anything. asked been for for guest...,11202
12,Jaime Lannister,"As I It's brother, duty feel it's much. my sho...",10823


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(stop_words='english')

In [15]:
embeddings = cv.fit_transform(new_df['words']).toarray()

In [16]:
embeddings.shape

(150, 16225)

In [17]:
embeddings = embeddings.astype('float64')

In [18]:
from sklearn.manifold import TSNE

In [19]:
tsne = TSNE(n_components=2, verbose=1, random_state=123)
z = tsne.fit_transform(embeddings)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 150 samples in 0.003s...
[t-SNE] Computed neighbors for 150 samples in 0.315s...
[t-SNE] Computed conditional probabilities for sample 150 / 150
[t-SNE] Mean sigma: 8.446655
[t-SNE] KL divergence after 250 iterations with early exaggeration: 69.383209
[t-SNE] KL divergence after 800 iterations: 0.520230


In [20]:
z.shape

(150, 2)

In [21]:
new_df['x'] = z.T[0]
new_df['y'] = z.T[1]
#new_df['z'] = z.T[2]

In [22]:
new_df

,character,words,num_words,x,y
17,Tyrion Lannister,It Mmh. Northern about girls. is say the they ...,25924,-1.492993,5.126384
13,Cersei Lannister,And And Casterly One Rock. When about afraid. ...,14294,-1.486942,4.897776
3,Jon Snow,Father's Go on. watching. And mother. yourBran...,11488,-1.290652,4.814851
20,Daenerys Targaryen,We've a and anything. asked been for for guest...,11202,-1.755230,4.409145
12,Jaime Lannister,"As I It's brother, duty feel it's much. my sho...",10823,-1.394697,4.736664
...,...,...,...,...,...
54,Rakharo,"Hash Khaleesi? addrivat mae, shafka zali zhey ...",174,1.683645,-5.158489
184,Lannister Soldier #3,Bring about! her Come You You're a be cold gon...,165,0.628235,-2.780954
324,Black Walder Rivers,But King North The and are bandits. come. craw...,159,0.536279,-2.642677
247,Rattleshirt,Don't I already crow. got need one two. Gut Ha...,157,0.300764,-2.512382


In [23]:
import plotly.express as px
fig = px.scatter(new_df.head(100), x="x", y="y", color="character")
fig.show()

In [24]:
import pickle

pickle.dump(new_df,open('data.pkl','wb'))